In [ ]:
import numpy as np
from gfsupg.solver import CartesianGeometry, FiniteElement1D, Scipy2DFEM
from gfsupg.solver import DeC, ImplicitEuler

from gfsupg.plotting import *

from gfsupg.problem import *

import time
import scipy.sparse as sp

import matplotlib.pyplot as plt

In [ ]:

#problem.T_fin = 1.
order=2

FEM1Dx = FiniteElement1D(order-1,"gaussLobatto","gaussLobatto")
FEM1Dy = FiniteElement1D(order-1,"gaussLobatto","gaussLobatto")
dec = DeC((order+1)//2,order,"gaussLobatto")
# dec = DeC(4,5,"gaussLobatto")


In [ ]:
# problem = SmoothVortexTestCase(is_long=False, pert_coeff=1e-3, pert_type="opt") #LinearAdvection("smooth_vortex_long")
# problem = StommelGyreTestCase(is_long=False, pert_coeff=1e-3, pert_type="num") #LinearAdvection("smooth_vortex_long")
#problem = SourceVortexTestCase(is_long=True) #LinearAdvection("smooth_vortex_long")
problem = CoriolisVortexTestCase(is_long=True)

In [ ]:

Ns = np.array([4,4], dtype=np.int32)

geom = CartesianGeometry(problem.xL,problem.xR, Ns, problem.geometry_folder, BC=problem.BC)

In [ ]:
FEM2D = Scipy2DFEM(geom,FEM1Dx, FEM1Dy, folder=problem.folderName)

In [ ]:
solver = ImplicitEuler(problem, FEM2D, dec, GF = False, stab = "SUPG", trick_second_der=False)

In [ ]:
A, B = solver.build_whole_matrices(0.05, 0.01)

In [ ]:
A.shape

In [ ]:
B.shape

In [ ]:
q_now  = dict()
q      = np.zeros((len(solver.problem.vars), solver.FEM2D.n_dof_tot)) 
for var in solver.problem.vars:
    q_now[var]  = np.zeros((1, solver.FEM2D.n_dof_tot))
    for i in range(1): #range(self.DeC.n_subNodes):
        q_now[var][i,:] = solver.ic_vect[var]


In [ ]:
size_array = sum(np.array([q_now[k].shape[1] for k in solver.problem.vars]))            
vect_q = np.empty((q_now['u'].shape[0], size_array))

solver.build_whole_q_vector(q_now, vect_q)

In [ ]:
vect_q.shape

In [ ]:
A @ vect_q[0,:]

In [ ]:
indices = [1,2,3]

In [ ]:
solver.problem.dirichlet

In [ ]:
from gfsupg.solver import Dirichlet_BC_set

In [ ]:
if solver.problem.dirichlet is not None:
    dirichlet_BC = dict()
    for bc_item in solver.problem.dirichlet.keys():
        BC_values = dict()
        idxs = solver.FEM2D.dirichlet_indexes[bc_item]
        for var in solver.problem.dirichlet[bc_item]:
            BC_values[var] = solver.ic_vect[var][idxs]
        dirichlet_BC[bc_item] = Dirichlet_BC_set(idxs, BC_values)

In [ ]:
A, B = solver.build_whole_matrices(0.01, 0.001, dirichlet_BC)

In [ ]:
A.todense()

In [ ]:
i = 0

In [ ]:
from gfsupg.solver import delete_row_in_coo_and_keep_diag_one, put_zero_row_in_coo

In [ ]:
delete_row_in_coo_and_keep_diag_one(A_small, i).todense()

In [ ]:
put_zero_row_in_coo(A_small, i).todense()

In [ ]:
idx_row = (A_small.row==i) & (A_small.col != i)
diag_i = (A_small.row==i) & (A_small.col == i)
idx_higher_rows = A_small.row>i

new_data = np.copy(A_small.data)
new_col = np.copy(A_small.col)
new_row = np.copy(A_small.row)

new_data[diag_i] = 1.0 
new_data=np.delete(new_data,idx_row)
new_col = np.delete(new_col, idx_row)
#new_row[idx_higher_rows]-=1
new_row= np.delete(new_row,idx_row)

In [ ]:
idx_row

In [ ]:
diag_i

In [ ]:
from scipy import sparse

In [ ]:
sparse.coo_matrix((new_data,(new_row,new_col)), shape = (A.shape[0],A.shape[1])).todense()

In [ ]:
A_small.col

In [ ]:
A_small.row